# 问题定位

前面四节走的都是"顺利路径"。这一节练一件事：**出错时怎么定位**。

云端版只能改配置、不能改源码（包里没有源码），所以本节聚焦两件事：
**按步骤定位**和**读日志**。想改源码重编，看 3.9（选看）。

## 本节学习大纲

- 先确定失败发生在哪一步
- 实战：故意写错配置，看它怎么报错
- 修复配置并验证
- 日志在哪里、怎么读
- 其他执行模式
- 遇到问题先看这里

## 1. 先确定失败发生在哪一步

一条完整的路径是：**启动平台 → 创建通信域 → 运行测试用例 → 检查结果**。
报错时，先找到这条路径中**第一个没有完成的步骤**，再读对应日志。

| 现象 | 优先检查什么 | 具体操作 |
| :--- | :--- | :--- |
| `start` 提示文件不存在 | 当前目录、配置名称 | 确认在 `~/hccl_vm_pack` 下，参数不带 `.yaml` |
| `mock-comm` 提示 YAML 格式错误 | 通信域配置字段 | 对照 03.06 检查 `meta`、`servers`、`ranks` 的写法 |
| 测试用例迟迟不能完成初始化 | 通信域人数与 MPI 进程数 | 数清配置里的参与设备，确认与 `mpirun -np` 一致 |
| 日志里没有 `check_result` | 测试用例是否执行到校验步骤 | 看有没有 `Entry-HcclCommInitClusterInfo` |
| 测试用例跑完但 Checker 报失败 | 任务依赖、读写地址 | 保存 Checker 完整日志，找第一个 `ErrorCode` 及其附近的任务信息 |

两个工具检查不同的东西：测试用例比较计算结果，Checker 还检查任务之间的依赖和内存访问。
**一次运行碰巧算对，也可能存在需要修复的任务冲突**——所以两个都要看。

## 2. 实战：故意写错配置

下面故意把 `ranks` 写成 `0`。这个字段需要一个**设备编号列表**，正确形式是 `[0, 1]`。

先创建这份错误配置（新文件名，不影响其他通信域）：

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

DEBUG_DIR="$VM_INSTALL/records/debug_demo"
mkdir -p "$DEBUG_DIR"

cat > "$VM_INSTALL/config/topo_meta/tutorial_debug_ranks.yaml" <<'YAML'
meta:
  podNum: 1
  serNum: 1
  rankNum: 2
topology:
  - podId: 0
    servers:
      - serId: 0
        ranks: 0
YAML

cp "$VM_INSTALL/config/topo_meta/tutorial_debug_ranks.yaml" "$DEBUG_DIR/bad_config.yaml"
cat "$DEBUG_DIR/bad_config.yaml"

`ranks: 0` 就是那个错误——`0` 是一个数字，表达不了"选设备 0 和设备 1"。

启动平台，然后试着用这份配置创建通信域：

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

tmux kill-server 2>/dev/null
# kill-server 是异步的：等旧服务端真正退出，否则 new-session 会撞上"正在死的 server"
for i in $(seq 1 20); do
    tmux list-sessions >/dev/null 2>&1 || break
    sleep 0.2
done
for i in 1 2 3; do
    tmux new-session -d -s hvm -x 200 -y 50 "$PACK/run_hccl_vm.sh start $CLUSTER_YAML 2>&1 | tee /tmp/hvm_start.log" && break
    sleep 1
done
if ! tmux has-session -t hvm 2>/dev/null; then
    echo "!! session not started, start output below:"
    tail -20 /tmp/hvm_start.log 2>/dev/null || true
fi
for i in $(seq 1 40); do
    tmux capture-pane -pt hvm 2>/dev/null | grep -q listening && break
    sleep 1
done

tmux send-keys -t hvm "$VM mock-comm tutorial_debug_ranks" Enter
sleep 8
tmux capture-pane -pt hvm | tail -6

期望在最后看到类似这样一行：

```
[error] ... YAML : 'ranks' must be a list.
```

**注意失败发生在哪一步**：配置解析阶段就停了，测试用例根本没启动。
所以这时候去翻测试用例日志是找不到原因的——方向错了。

> 中间的 `warning`（`topo.json` 已存在、将被覆盖）是每次创建通信域的正常现象，不是错误。

## 3. 修复配置并验证

把 `ranks: 0` 改成 `ranks: [0, 1]`，再创建一次通信域。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

sed -i 's/ranks: 0/ranks: [0, 1]/' "$VM_INSTALL/config/topo_meta/tutorial_debug_ranks.yaml"
cp "$VM_INSTALL/config/topo_meta/tutorial_debug_ranks.yaml" "$VM_INSTALL/records/debug_demo/fixed_config.yaml"

tmux send-keys -t hvm "$VM mock-comm tutorial_debug_ranks" Enter
sleep 8
tmux capture-pane -pt hvm | tail -4

这次应该看到 `config by tutorial_debug_ranks.yaml`，不再有 `must be a list`。

**但"没报错"不等于修好了。** 用 `rank_count` 验证——它表示本次通信域共有几个 rank：

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
grep -n '"rank_count"' "$VM_INSTALL/data/ranktable.json"

期望 `"rank_count": 2`，和 YAML 里 `rankNum: 2` 对得上。

这一步体现的是定位的基本节奏：**报错 → 改 → 用可量化的字段验证**，而不是"没报错就过了"。

## 4. 日志在哪里、怎么读

云端版的日志分三类，位置都在包里：

| 日志 | 位置 | 看什么 |
| :--- | :--- | :--- |
| 会话屏幕 | `tmux capture-pane -pt hvm` | 命令回显、`[error]`、Checker 汇总 |
| 测试用例输出 | 你重定向的那个文件（如 `$VM_INSTALL/bin/log.txt`） | `check_result` 表、`Entry-Hccl*` |
| 组件日志 | `$VM_INSTALL/logs/`（`proxy/`、`checker/`） | 更细的模块级报错 |

常用几条筛选：

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh

echo "=== error 计数 ==="
grep -c "\[error\]" "$VM_INSTALL/bin/log.txt" 2>/dev/null || echo "no log.txt"
echo "=== 去重后的 error ==="
grep "\[error\]" "$VM_INSTALL/bin/log.txt" 2>/dev/null | sed 's/.*\]\[[0-9]*\]//' | sort -u | head -5
echo "=== HCCL 入口 ==="
grep -o "Entry-Hccl[A-Za-z]*" "$VM_INSTALL/bin/log.txt" 2>/dev/null | sort | uniq -c

三条命令分别回答三个问题：

- **有多少条 error** —— 先看量级，几十条和 1 条的处理方式完全不同；
- **去重后是什么** —— 同一条错误会被多个 rank 各打一遍，去重才看得清；
- **算子有没有被调用** —— 没有 `Entry-Hccl*` 说明测试用例在初始化阶段就退出了。

> 读日志的顺序：**先看最后一个"成功"的步骤，再看它之后的第一个 error**。
> 从中间随便挑一条 error 去看，容易被无关的告警带偏。

## 5. 其他执行模式

`hccl-vm run` 提供两个省资源的模式，适合大集群试跑：

| 模式 | 作用 | 适用场景 |
| :--- | :--- | :--- |
| `--check-only` | 大块内存（200 MB~4 GB）复用同一块 4 GB 共享区，内容不保证正确 | 内存不够、只想看任务图能不能建起来 |
| `--estimate` | server 侧合成幻影对端，跳过真实对端建链阻塞，只生成任务清单 | 大集群拓扑验证，不真跑数据传输 |

用法（把参数加在 `run` 的 yaml 名后面）：

```bash
$PACK/run_hccl_vm.sh run ascend950_cluster_32_server_normal --check-only <测试用例命令>
```

> 这两个模式只在 `run`（一次性模式）里有效，`start` 会话里没有。

## 6. 想改源码重编怎么办

云端包里**没有源码**，所以下面这些事在云端做不到：

- 改 `hccl_test` 的测试逻辑
- 改 Checker 的校验规则
- 改 HCCL-VM 的模拟行为
- 改完重新编译、重新部署

需要做这些时，走 **3.9 的本地环境**：在 WSL 里装 CANN、编译 HCCL-VM、
改源码、`build_pkg.sh` 重新部署，再打包成新的绿色包。

原课件的「改错一条依赖 → 读 Checker 定位信息 → 恢复源码」这一整套练习，
就是 3.9 的内容。

## 7. 遇到问题先看这里

| 现象 | 原因 | 处理 |
| :--- | :--- | :--- |
| `mock-comm` 报 `'ranks' must be a list` | YAML 里 `ranks` 写成了数字 | 改成列表 `[0, 1]` |
| 改完配置仍然报同样的错 | 改的是另一个文件，或没保存 | `cat` 一下那个 YAML 确认内容 |
| `rank_count` 和 `rankNum` 不一致 | 配置里 `topology` 的设备数与 `meta` 不符 | 对照检查两处 |
| `grep -c` 输出 0 但确实失败了 | 日志路径不对（`log.txt` 不在 `bin/`） | 确认测试用例命令里的重定向路径 |
| 想看清某条 error 的上下文 | 只有一行看不出来 | 用 `grep -n -A 5 "关键字" 日志文件` |

[← 3.7 通信用例实战](./03.07_communication_cases.ipynb) · [返回本章目录](./03.01_chapter_intro.ipynb) · [3.9 本地 WSL 环境与自建绿色包（选看） →](./03.09_appendix_local_env.ipynb)